# AAT6153 Chapter 1 — Hands-on 3
## Pixels, Convolutions and Pooling
**Slides:** Introduction to Computer Vision (slides 27–32), Understanding Convolutions (slides 38–44), Typical CNN Architecture (slides 45–49)

In this notebook you will:
1. Treat an image as a **NumPy array**: its shape, pixels, channels and coordinate system.
2. Implement **2-D convolution** (cross-correlation) yourself, with **stride** and **zero-padding**.
3. Apply hand-designed kernels for **blurring, sharpening and edge detection**.
4. Show that a Keras `Conv2D` layer does the same operation, but **learns** its kernels.
5. Compare the **parameter counts** of fully-connected and convolutional layers, and try **max pooling**.

*Runtime: CPU is fine for this notebook.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from skimage import data

---
# Part A — Images are arrays of pixels (slides 27–32)

In [ ]:
image = data.astronaut()          # a built-in RGB test image
print("Shape (H, W, D):", image.shape)
print("Data type:      ", image.dtype)
print("Total pixels:   ", image.shape[0] * image.shape[1])

plt.imshow(image); plt.title("RGB image"); plt.axis(False); plt.show()

### Image coordinate system (slides 30–32)
The origin (0, 0) is the **upper-left** corner. NumPy indexes as `image[row, column]` = `image[y, x]`.

In [ ]:
print("Pixel at x=300, y=15  →", image[15, 300])   # [R, G, B]
print("Pixel at x=50,  y=250 →", image[250, 50])

plt.imshow(image)
for (x, y) in [(300, 15), (50, 250)]:
    plt.scatter(x, y, s=120, c='yellow', edgecolors='k')
    plt.annotate(f"(x={x}, y={y})", (x, y), xytext=(x + 15, y + 25), color='yellow', fontsize=11)
plt.title("x increases to the right, y increases downward")
plt.show()

### Colour channels (slides 28–29, 31)
An RGB image is a stack of three 2-D arrays. A **grayscale** image has a single channel with values 0–255.
**OpenCV stores channels in BGR order**. If you forget to convert, red and blue are swapped.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for i, (ch, cmap) in enumerate(zip("RGB", ["Reds", "Greens", "Blues"])):
    axes[i].imshow(image[:, :, i], cmap=cmap); axes[i].set_title(f"{ch} channel")

gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
axes[3].imshow(gray, cmap='gray'); axes[3].set_title(f"Grayscale {gray.shape}")

bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
axes[4].imshow(bgr); axes[4].set_title("BGR shown as RGB (wrong!)")
for ax in axes: ax.axis(False)
plt.tight_layout(); plt.show()

---
# Part B — Convolution from scratch (slides 38–42)
Deep learning libraries implement convolution as **cross-correlation**:

$$S(i, j) = \sum_m \sum_n I(i + m,\ j + n)\, K(m, n)$$

The kernel slides left-to-right and top-to-bottom over the image with **stride** $S$. **Zero-padding** $P$ adds a border of zeros so the kernel can be applied at the edges.
The output size is

$$\text{out} = \left\lfloor \frac{W - K + 2P}{S} \right\rfloor + 1$$

In [ ]:
def conv2d(image, kernel, stride=1, padding=0):
    """2-D cross-correlation of a single-channel image with a kernel."""
    if padding > 0:
        image = np.pad(image, padding, mode='constant', constant_values=0)
    kh, kw = kernel.shape
    out_h = (image.shape[0] - kh) // stride + 1
    out_w = (image.shape[1] - kw) // stride + 1
    output = np.zeros((out_h, out_w))
    for y in range(out_h):
        for x in range(out_w):
            region = image[y * stride:y * stride + kh, x * stride:x * stride + kw]
            output[y, x] = np.sum(region * kernel)     # element-wise multiply, then sum
    return output

Let's reproduce the example from slide 40: a 5 × 5 binary image and a 3 × 3 kernel.

In [ ]:
I = np.array([[1, 1, 1, 0, 0],
              [0, 1, 1, 1, 0],
              [0, 0, 1, 1, 1],
              [0, 0, 1, 1, 0],
              [0, 1, 1, 0, 0]])
K = np.array([[1, 0, 1],
              [0, 1, 0],
              [1, 0, 1]])

print("No padding, stride 1  →  output shape", conv2d(I, K).shape)
print(conv2d(I, K).astype(int))
print("\nZero-padding 1, stride 1  →  output shape", conv2d(I, K, padding=1).shape)
print(conv2d(I, K, padding=1).astype(int))
print("\nZero-padding 1, stride 2  →  output shape", conv2d(I, K, stride=2, padding=1).shape)
print(conv2d(I, K, stride=2, padding=1).astype(int))

**Check your understanding:** use the output-size formula to confirm the three shapes above (W = 5, K = 3).

---
# Part C — Hand-designed kernels (slides 42–43)
Classic image processing uses kernels that are **designed by hand** for each task.

In [ ]:
kernels = {
    "Box blur (7×7)": np.ones((7, 7)) / 49,
    "Sharpen": np.array([[0, -1, 0],
                         [-1, 5, -1],
                         [0, -1, 0]]),
    "Laplacian (edges)": np.array([[0, 1, 0],
                                   [1, -4, 1],
                                   [0, 1, 0]]),
    "Sobel X (vertical edges)": np.array([[-1, 0, 1],
                                          [-2, 0, 2],
                                          [-1, 0, 1]]),
    "Sobel Y (horizontal edges)": np.array([[-1, -2, -1],
                                            [0, 0, 0],
                                            [1, 2, 1]]),
}

small = cv2.resize(gray, (256, 256)).astype('float32')   # smaller image so our Python loop is quick

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.ravel()
axes[0].imshow(small, cmap='gray'); axes[0].set_title("Original")
for ax, (name, k) in zip(axes[1:], kernels.items()):
    out = conv2d(small, k, padding=k.shape[0] // 2)          # 'same' padding
    ax.imshow(np.abs(out) if "Sobel" in name or "Laplacian" in name else out, cmap='gray')
    ax.set_title(name)
for ax in axes: ax.axis(False)
plt.tight_layout(); plt.show()

In [ ]:
# Our implementation agrees with OpenCV's filter2D (which is also cross-correlation)
k = kernels["Sobel X (vertical edges)"].astype('float32')
ours = conv2d(small, k, padding=1)
opencv = cv2.filter2D(small, -1, k, borderType=cv2.BORDER_CONSTANT)
print("Max difference vs OpenCV:", np.abs(ours - opencv).max())

---
# Part D — From hand-designed to learned kernels (slides 43–44)
A Keras `Conv2D` layer performs exactly the same operation. Below we **manually set** its kernel to Sobel X and confirm it gives the same result.
In a CNN, these kernel values are **not** set by hand: they are **learned by backpropagation**. You will see learned kernels in Hands-on 4.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

conv = layers.Conv2D(filters=1, kernel_size=(3, 3), padding='same', use_bias=False)
conv.build((None, 256, 256, 1))
conv.set_weights([k.reshape(3, 3, 1, 1)])           # (kh, kw, in_channels, out_channels)

keras_out = conv(small.reshape(1, 256, 256, 1)).numpy().squeeze()
print("Max difference vs our conv2d:", np.abs(keras_out - ours).max())

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(np.abs(ours), cmap='gray'); axes[0].set_title("Our conv2d — Sobel X")
axes[1].imshow(np.abs(keras_out), cmap='gray'); axes[1].set_title("Keras Conv2D — Sobel X")
for ax in axes: ax.axis(False)
plt.show()

---
# Part E — Why CNNs need far fewer parameters (slides 45–46)
Compare a fully-connected hidden layer of 100 neurons with a convolutional layer of 100 kernels of size 5 × 5, both on a 250 × 250 grayscale image.

In [ ]:
from tensorflow.keras.models import Sequential

fc = Sequential([layers.Input(shape=(250 * 250,)), layers.Dense(100)])
cnn = Sequential([layers.Input(shape=(250, 250, 1)), layers.Conv2D(100, (5, 5))])

print(f"Fully-connected (100 neurons):  {fc.count_params():>10,} parameters")
print(f"Convolutional  (100 kernels):   {cnn.count_params():>10,} parameters")
print(f"Ratio: {fc.count_params() / cnn.count_params():,.0f}× fewer for the CNN layer")

The conv layer re-uses the **same small kernel** at every position in the image (weight sharing), so its parameter count does not depend on the image size.

---
# Part F — Max pooling (slide 49)
A 2 × 2 max pool with stride 2 keeps the largest value in each 2 × 2 window, halving the width and height.

In [ ]:
x = np.array([[1, 1, 2, 4],
              [5, 6, 7, 8],
              [3, 2, 1, 0],
              [1, 2, 3, 4]], dtype='float32')

pooled = layers.MaxPooling2D(pool_size=(2, 2), strides=2)(x.reshape(1, 4, 4, 1)).numpy().squeeze()
print("Input:\n", x.astype(int))
print("\nAfter 2×2 max pooling, stride 2:\n", pooled.astype(int))

In [ ]:
# Pooling an edge map: the important structure survives at a quarter of the size
edges = np.abs(ours).reshape(1, 256, 256, 1).astype('float32')
pooled_edges = layers.MaxPooling2D(2, 2)(edges).numpy().squeeze()

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(edges.squeeze(), cmap='gray'); axes[0].set_title(f"Edge map {edges.shape[1:3]}")
axes[1].imshow(pooled_edges, cmap='gray'); axes[1].set_title(f"After max pooling {pooled_edges.shape}")
for ax in axes: ax.axis(False)
plt.show()

---
# Part G — Try it yourself
1. Design a kernel that detects **diagonal** edges and apply it to the image.
2. Apply the 7 × 7 box blur with `stride=2` and no padding. Predict the output size with the formula first.
3. Load your **own photo** (`from google.colab import files; files.upload()`, then `cv2.imread` and remember **BGR → RGB**) and repeat Part C.
4. In Part E, change the image to 500 × 500. How do the two parameter counts change?

In [ ]:
# Use this cell for your experiments